In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree


class NitroReduction(MorphingOperator):
    def __init__(self):
        super(NitroReduction, self).__init__()
        self._name = "Nitro Reduction (Phase I - Safe)"
        self._target_nitrogens = []
        self.NITRO_PATTERN = Chem.MolFromSmarts("[C,c][N;X3](=[O,O-])~[O,O-]")

    def setOriginal(self, mol):
        super(NitroReduction, self).setOriginal(mol)
        self._target_nitrogens = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.NITRO_PATTERN)
        for match in matches:
            if match[1] not in self._target_nitrogens:
                self._target_nitrogens.append(match[1])

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_nitrogens:
            return MolpherMol(other=rdkit_mol)

        n_idx = random.choice(self._target_nitrogens)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            n_atom = rw_mol.GetAtomWithIdx(n_idx)
            
            o_indices = [neighbor.GetIdx() for neighbor in n_atom.GetNeighbors() if neighbor.GetAtomicNum() == 8]
            
            for o_idx in o_indices:
                bond = rw_mol.GetBondBetweenAtoms(n_idx, o_idx)
                if bond:
                    rw_mol.RemoveBond(n_idx, o_idx)
            
            n_atom.SetFormalCharge(0)
            n_atom.SetNoImplicit(False)
            n_atom.SetNumExplicitHs(2)
            for prop in list(n_atom.GetPropNames()):
                n_atom.ClearProp(prop)
            
            new_mol = rw_mol.GetMol()
            new_mol.UpdatePropertyCache(strict=False)
            
            atoms_to_remove = []
            for atom in new_mol.GetAtoms():
                if atom.GetAtomicNum() == 8 and atom.GetDegree() == 0:
                    atoms_to_remove.append(atom.GetIdx())

            edit = Chem.RWMol(new_mol)
            for idx in sorted(atoms_to_remove, reverse=True):
                edit.RemoveAtom(idx)
            new_mol = edit.GetMol()

            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            return MolpherMol(other=new_mol)

        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class NAcetylation(MorphingOperator):
    def __init__(self):
        super(NAcetylation, self).__init__()
        self._name = "N-acetylation (Primary Amines)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")

    def setOriginal(self, mol):
        super(NAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            chosen_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(chosen_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [chosen_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

nitro_op = NitroReduction()
nacet_op = NAcetylation()

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
        
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target
closest_info = FindClosest()

start_mol = MolpherMol("CC(=O)Nc1ccc([N+](=O)[O-])cc1") 
target_mol = MolpherMol("CC(=O)Nc1ccc(NC(C)=O)cc1")
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (nitro_op, nacet_op)
max_generations = 40

while not tree.path_found and tree.generation_count < max_generations:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
    
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    if closest_info.closest_mol:
        print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)

Generation #1
Molecules in tree: 2
Closest to target: CC(=O)NC1=CC=C(N)C=C1 (Distance: 0.2500)
----------------------------------------
Generation #2
Molecules in tree: 3
Closest to target: CC(=O)NC1=CC=C(NC(C)=O)C=C1 (Distance: 0.0000)
----------------------------------------


In [2]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree


class NitroReduction(MorphingOperator):
    def __init__(self):
        super(NitroReduction, self).__init__()
        self._name = "Nitro Reduction (Phase I - Safe)"
        self._target_nitrogens = []
        self.NITRO_PATTERN = Chem.MolFromSmarts("[C,c][N;X3](=[O,O-])~[O,O-]")

    def setOriginal(self, mol):
        super(NitroReduction, self).setOriginal(mol)
        self._target_nitrogens = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.NITRO_PATTERN)
        for match in matches:
            if match[1] not in self._target_nitrogens:
                self._target_nitrogens.append(match[1])

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._target_nitrogens:
            return MolpherMol(other=rdkit_mol)

        n_idx = random.choice(self._target_nitrogens)

        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            n_atom = rw_mol.GetAtomWithIdx(n_idx)
            
            o_indices = [neighbor.GetIdx() for neighbor in n_atom.GetNeighbors() if neighbor.GetAtomicNum() == 8]
            
            for o_idx in o_indices:
                bond = rw_mol.GetBondBetweenAtoms(n_idx, o_idx)
                if bond:
                    rw_mol.RemoveBond(n_idx, o_idx)
            
            n_atom.SetFormalCharge(0)
            n_atom.SetNoImplicit(False)
            n_atom.SetNumExplicitHs(2)
            for prop in list(n_atom.GetPropNames()):
                n_atom.ClearProp(prop)
            
            new_mol = rw_mol.GetMol()
            new_mol.UpdatePropertyCache(strict=False)
            
            atoms_to_remove = []
            for atom in new_mol.GetAtoms():
                if atom.GetAtomicNum() == 8 and atom.GetDegree() == 0:
                    atoms_to_remove.append(atom.GetIdx())

            edit = Chem.RWMol(new_mol)
            for idx in sorted(atoms_to_remove, reverse=True):
                edit.RemoveAtom(idx)
            new_mol = edit.GetMol()

            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            return MolpherMol(other=new_mol)

        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class NAcetylation(MorphingOperator):
    def __init__(self):
        super(NAcetylation, self).__init__()
        self._name = "N-acetylation (Primary Amines)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")

    def setOriginal(self, mol):
        super(NAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            chosen_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(chosen_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [chosen_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

nitro_op = NitroReduction()
nacet_op = NAcetylation()

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
        
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target
closest_info = FindClosest()

start_mol  = MolpherMol("O=S(=O)(Nc1nc(C)cc(C)n1)c1ccc([N+](=O)[O-])cc1")
target_mol = MolpherMol("CC(=O)Nc1ccc(S(=O)(=O)Nc2nc(C)cc(C)n2)cc1")
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (nitro_op, nacet_op)
max_generations = 40

while not tree.path_found and tree.generation_count < max_generations:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
    
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    if closest_info.closest_mol:
        print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)

Generation #1
Molecules in tree: 2
Closest to target: CC1=CC(C)=NC(NS(=O)(=O)C2=CC=C(N)C=C2)=N1 (Distance: 0.3696)
----------------------------------------
Generation #2
Molecules in tree: 3
Closest to target: CC(=O)NC1=CC=C(S(=O)(=O)NC2=NC(C)=CC(C)=N2)C=C1 (Distance: 0.0000)
----------------------------------------
